In [ ]:
import src.utils as ut
import src.datasets as ds
from cdlib import algorithms
import networkx as nx

import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx, to_undirected
from torch_geometric.data import Data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [1]:
datasplit = ds.DataSplit("wiki_science", device, 1)
data, split = datasplit.get(0)

NameError: name 'ds' is not defined

In [ ]:
G = to_networkx(data, to_undirected=True)
communities = algorithms.louvain(G).communities

color_map = {}
for i, community in enumerate(communities):
    for node in community:
        color_map[node] = i
node_colors = [color_map[node] for node in G.nodes()]

if hasattr(data, "_pos"):
    pos = {}
    for i, p in enumerate(data._pos):
        x, y = p.split(", ")
        pos[i] = np.array([float(x), float(y)])
else:
    pos = nx.spring_layout(G)

In [ ]:
def plot(data, commu, title= ""):
    G = to_networkx(data, to_undirected=True)
    color_map = {}
    for i, community in enumerate(commu):
        for node in community:
            color_map[node] = i
    node_colors = [color_map[node] for node in G.nodes()]
    plt.figure(figsize=(20, 17))
    nx.draw(G, node_size=30, pos=pos, node_color=node_colors, cmap=plt.cm.get_cmap('jet', len(commu)))
    # nx.draw_networkx_labels(G, pos, font_size=12, font_color='black')
    plt.title(title)
    plt.show()
    print(data.edge_index.shape)
plot(data, communities, "graphe d'entrée")

In [ ]:
from torch_geometric.nn import Node2Vec
model = Node2Vec(
    data.edge_index,
    embedding_dim=128,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    p=1.0,
    q=1.0,
    sparse=True,
).to(device)

num_workers = 4
loader = model.loader(batch_size=128, shuffle=True, num_workers=num_workers)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

def train():
    model.train()
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)



for epoch in range(1, 101):
    loss = train()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

In [ ]:
emb = torch.tensor(model())

In [ ]:
from scgravity import filter_data, create_q_bin, calculate_mass
import torch.nn as nn
from tqdm import tqdm

cos = nn.CosineSimilarity(dim=0)
dist_data = {}
for u in tqdm(range(data.num_nodes)):
    for v in range(data.num_nodes):
        if str(u) in dist_data.keys():
            if str(v) not in dist_data[str(u)]:
                dist_uv = cos(emb[u], emb[v]).item()
                dist_data[str(u)].update({str(v): dist_uv})
                if str(v) in dist_data.keys():
                    dist_data[str(v)].update({str(u): dist_uv})
        else:
            dist_uv = cos(emb[u], emb[v]).item()
            dist_data[str(u)] = {str(v): dist_uv}
            dist_data[str(v)] = {str(u): dist_uv}

display(dist_data)

In [ ]:
od_data = {}
for edge in data.edge_index.T:
    u = int(edge[0])
    v = int(edge[1])
    if str(u) in od_data.keys():
        od_data[str(u)].update({str(v): 1})
    else:
        od_data[str(u)] = {str(v): 1}
display(od_data)

In [ ]:
od_data_clean = filter_data(od_data, dist_data)
q_bin = create_q_bin(od_data_clean, dist_data, each_num=500)
m_in, m_out, Q_hist, Q_std = calculate_mass(od_data_clean, q_bin)

In [ ]:
import matplotlib.pyplot as plt

bin_mid = q_bin["bin_mid"]  # midpoints of each distance bin

plt.figure(figsize=(7,4))
plt.plot(bin_mid, Q_hist, marker='o')
plt.xlabel("Distance (d)")
plt.ylabel("Q(d)")
plt.title("Estimated Deterrence Function Q(d)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
f_est_ij = m_out["1"] * m_in["2"] * Q_hist[q_bin["call_dic"]["1"]["2"]]